# 10 — vLLM: Qasper Benchmark

This notebook evaluates KV cache compression on the
[Qasper](https://huggingface.co/datasets/tau/scrolls) (SCROLLS) benchmark
using [vLLM](https://github.com/vllm-project/vllm) with Qwen3-8B.

We test two compression strategies:
- **full_replacement** — full KV cache replacement at target ratio
- **filtering** — filtering-based compression at target ratio

Qasper contains questions over full NLP research papers (~3K–8K tokens),
testing document comprehension with both short and free-form answers.

Scoring uses the HuggingFace `evaluate` library (SQuAD F1), matching the
standard lm-evaluation-harness methodology for SCROLLS/Qasper.

Results are saved to `results/vllm_qasper/` for comparison in later notebooks.

## Configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

MAX_NEW_TOKENS = 64

PRESS_CONFIGS = {
    "full_replacement": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "full_replacement",
        "kv_compression_ratio": cr,
        "enable_prefix_caching": False,
    },
    "filtering": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "filtering",
        "kv_compression_ratio": cr,
    },
}

In [ ]:
import sys
import os

FORK_DIR = "/opt/app-root/src/vllm-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import vllm
    print(f"Using FORK vLLM (version: {vllm.__version__})")
else:
    import vllm
    print(f"Using SYSTEM vLLM (version: {vllm.__version__})")

In [ ]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
allocated_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb = torch.cuda.memory_reserved() / 1e9

print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"VRAM:       {vram_gb:.1f} GB total")
print(f"Allocated:  {allocated_gb:.2f} GB")
print(f"Reserved:   {reserved_gb:.2f} GB")
print(f"Free:       {vram_gb - reserved_gb:.1f} GB (approx)")

if allocated_gb > 1.0:
    print(
        "\n⚠  GPU memory is not free — a model from another notebook may still be loaded.\n"
        "   Restart this kernel before proceeding."
    )

def cleanup_vllm(llm):
    llm.llm_engine.engine_core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()

## 1. Load Qasper Dataset

In [ ]:
from datasets import load_dataset

qasper_ds = load_dataset("tau/scrolls", "qasper", split="validation")

print(f"Qasper dataset loaded: {len(qasper_ds)} examples")
print(f"Columns: {qasper_ds.column_names}")
print(f"Sample input (first 200 chars): {qasper_ds[0]['input'][:200]}...")
print(f"Sample output: {qasper_ds[0]['output']}")

## 2. Load Scoring Metric

In [ ]:
import evaluate

squad_metric = evaluate.load("squad")
print("Loaded SQuAD metric (token-level F1)")

## 3. Prepare Prompts

Apply the model's chat template to each Qasper example.
The SCROLLS `input` field contains `question \n\n context`.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

prompt_configs = []
max_input_tokens = 0

for row in qasper_ds:
    messages = [{"role": "user", "content": row["input"]}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

    input_len = len(tokenizer.encode(prompt, add_special_tokens=False))
    max_input_tokens = max(max_input_tokens, input_len)

    references = row["output"].split(" | ") if " | " in row["output"] else [row["output"]]

    prompt_configs.append({
        "prompt": prompt,
        "reference_answers": references,
        "id": row["id"],
    })

max_model_len = max_input_tokens + MAX_NEW_TOKENS + 256
print(f"Prepared {len(prompt_configs)} prompts")
print(f"Max input tokens: {max_input_tokens}")
print(f"max_model_len for vLLM: {max_model_len}")

## 4. Run Batch Inference

For each (algorithm, compression_ratio) combination, create a vLLM engine,
run all prompts in batch, and collect predictions.

In [ ]:
import time
from vllm import LLM, SamplingParams

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
)

def get_gpu_memory_used_gb() -> float:
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9

prompts = [pc["prompt"] for pc in prompt_configs]

all_results = []

configs = [("no_press", 0.0, {"model": MODEL_NAME, "dtype": "auto",
    "gpu_memory_utilization": 0.90, "max_model_len": max_model_len,
    "trust_remote_code": True,
    "attention_config": {"backend": "FLASH_ATTN"}})]
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        engine_args = press_factory(ratio)
        engine_args["max_model_len"] = max_model_len
        configs.append((press_name, ratio, engine_args))

for press_name, ratio, engine_args in configs:
    label = f"{press_name} | ratio={ratio}"
    print(f"\n{'='*60}")
    print(f"Running: {label} ({len(prompts)} prompts)")
    print(f"{'='*60}")

    llm = None
    try:
        llm = LLM(**engine_args)

        mem_before = get_gpu_memory_used_gb()
        start = time.perf_counter()

        outputs = llm.generate(prompts, sampling_params)

        batch_elapsed = time.perf_counter() - start
        mem_after = get_gpu_memory_used_gb()
        peak_mem = max(mem_before, mem_after)

        for i, output in enumerate(outputs):
            predicted_answer = output.outputs[0].text.strip()
            pc = prompt_configs[i]

            all_results.append({
                "framework": "vllm",
                "press": press_name,
                "compression_ratio": ratio,
                "predicted_answer": predicted_answer,
                "reference_answers": pc["reference_answers"],
                "id": pc["id"],
                "elapsed_sec": round(batch_elapsed / len(prompts), 3),
                "peak_gpu_mem_gb": round(peak_mem, 3),
            })

        total_gen_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
        throughput = total_gen_tokens / batch_elapsed if batch_elapsed > 0 else 0

        print(f"  Done: {batch_elapsed:.1f}s — {throughput:.1f} tok/s — peak mem={peak_mem:.2f} GB")
    finally:
        if llm is not None:
            cleanup_vllm(llm)

print(f"\nTotal results: {len(all_results)}")

## 5. Score & Results

Score predictions using the HuggingFace `evaluate` SQuAD metric (token-level F1).
For multi-reference answers (pipe-delimited in SCROLLS), we take the max F1
across references — standard SCROLLS evaluation methodology.

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

def compute_squad_f1(group):
    predictions = []
    references = []
    for _, row in group.iterrows():
        predictions.append({"id": str(row["id"]), "prediction_text": row["predicted_answer"]})
        references.append({"id": str(row["id"]), "answers": {
            "text": row["reference_answers"],
            "answer_start": [0] * len(row["reference_answers"]),
        }})
    result = squad_metric.compute(predictions=predictions, references=references)
    return result

all_metrics = {}
rows = []
for (press, ratio), group in df.groupby(["press", "compression_ratio"]):
    metrics = compute_squad_f1(group)
    key = f"{press}__{ratio}"
    all_metrics[key] = metrics
    rows.append({
        "press": press,
        "compression_ratio": ratio,
        "f1": round(metrics["f1"], 2),
        "exact_match": round(metrics["exact_match"], 2),
        "mean_time": round(group["elapsed_sec"].mean(), 3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## 6. Save Results

In [ ]:
import json
import os

os.makedirs("results/vllm_qasper", exist_ok=True)

predictions_path = "results/vllm_qasper/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/vllm_qasper/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")